In [1]:
!pip install -q transformers datasets rouge-score sentencepiece

  Preparing metadata (setup.py) ... done


In [2]:
from datasets import load_dataset

dataset = load_dataset("cnn_dailymail", "3.0.0")
print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

...

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 11490
    })
})


In [3]:
print(dataset)
print("\n--- Exemple ---")
print("Article:", dataset["train"][0]["article"][:300])
print("\nRésumé:", dataset["train"][0]["highlights"])

DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 11490
    })
})

--- Exemple ---
Article: LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappoi

Résumé: Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday .
Young actor says he has no plans to fritter his cash away .
Radcliffe's earnings from first five Potter films have been held in trust fund .


In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("t5-small")

# T5 nécessite un préfixe "summarize: " pour la tâche de résumé
def preprocess(examples):
    inputs = ["summarize: " + art for art in examples["article"]]
    model_inputs = tokenizer(
        inputs,
        max_length=512,
        truncation=True,
        padding="max_length"
    )
    labels = tokenizer(
        examples["highlights"],
        max_length=128,
        truncation=True,
        padding="max_length"
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# On prend un sous-ensemble pour accélérer l'entraînement
train_dataset = dataset["train"].select(range(5000))
val_dataset = dataset["validation"].select(range(500))

train_tokenized = train_dataset.map(preprocess, batched=True, remove_columns=["article", "highlights", "id"])
val_tokenized = val_dataset.map(preprocess, batched=True, remove_columns=["article", "highlights", "id"])

print(train_tokenized)
print(val_tokenized)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 5000
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 500
})


In [5]:
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

# Charge le modèle T5
model = AutoModelForSeq2SeqLM.from_pretrained("t5-small")

# Data collator
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# Configuration de l'entraînement
training_args = Seq2SeqTrainingArguments(
    output_dir="./t5-summarization",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    predict_with_generate=True,
    fp16=True,
    logging_dir="./logs",
)



model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [7]:
# Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    processing_class=tokenizer,
    data_collator=data_collator,
)

# Lance l'entraînement
trainer.train()

Epoch,Training Loss,Validation Loss
1,1.500444,0.840252
2,1.090591,0.840933
3,1.068208,0.840418


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=1875, training_loss=1.1893430826822917, metrics={'train_runtime': 438.1495, 'train_samples_per_second': 34.235, 'train_steps_per_second': 4.279, 'total_flos': 2030127022080000.0, 'train_loss': 1.1893430826822917, 'epoch': 3.0})

In [8]:
from rouge_score import rouge_scorer
import numpy as np

# Génère des résumés sur le set de validation
def generate_summary(text):
    inputs = tokenizer("summarize: " + text, return_tensors="pt", max_length=512, truncation=True).to(model.device)
    outputs = model.generate(inputs["input_ids"], max_length=128, num_beams=4, early_stopping=True)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Évalue sur 100 exemples
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
scores = {"rouge1": [], "rouge2": [], "rougeL": []}


In [9]:
for i in range(100):
    article = val_dataset[i]["article"]
    reference = val_dataset[i]["highlights"]
    prediction = generate_summary(article)

    result = scorer.score(reference, prediction)
    scores["rouge1"].append(result["rouge1"].fmeasure)
    scores["rouge2"].append(result["rouge2"].fmeasure)
    scores["rougeL"].append(result["rougeL"].fmeasure)

print("=== Résultats ROUGE ===")
print(f"ROUGE-1 : {np.mean(scores['rouge1']):.4f}")
print(f"ROUGE-2 : {np.mean(scores['rouge2']):.4f}")
print(f"ROUGE-L : {np.mean(scores['rougeL']):.4f}")

=== Résultats ROUGE ===
ROUGE-1 : 0.3159
ROUGE-2 : 0.1206
ROUGE-L : 0.2236


In [10]:
# Test avec un article financier
article = """
The Federal Reserve raised interest rates by 0.25 percentage points on Wednesday,
marking the tenth consecutive rate hike as the central bank continues its fight against
inflation. Fed Chair Jerome Powell said the decision was unanimous among policymakers.
The rate now stands at 5.25%, the highest level in 16 years. Markets reacted negatively,
with the S&P 500 falling 1.2% following the announcement. Analysts warn that further
hikes could push the economy into recession, while others believe the Fed is close to
pausing its tightening cycle. Inflation currently stands at 4.9%, still well above the
Fed's 2% target.
"""

summary = generate_summary(article)


In [11]:
print("=== Article original ===")
print(article)


=== Article original ===

The Federal Reserve raised interest rates by 0.25 percentage points on Wednesday, 
marking the tenth consecutive rate hike as the central bank continues its fight against 
inflation. Fed Chair Jerome Powell said the decision was unanimous among policymakers. 
The rate now stands at 5.25%, the highest level in 16 years. Markets reacted negatively, 
with the S&P 500 falling 1.2% following the announcement. Analysts warn that further 
hikes could push the economy into recession, while others believe the Fed is close to 
pausing its tightening cycle. Inflation currently stands at 4.9%, still well above the 
Fed's 2% target.



In [12]:
print("\n=== Résumé automatique ===")
print(summary)


=== Résumé automatique ===
Federal Reserve raises interest rates by 0.25 percentage points. The rate now stands at 5.25%, the highest level in 16 years. Markets reacted negatively, with the S&P 500 falling 1.2%.


In [13]:
model.save_pretrained("./t5-summarization-model")
tokenizer.save_pretrained("./t5-summarization-model")
print("Modèle sauvegardé ✅")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modèle sauvegardé ✅


In [14]:
import shutil
from google.colab import files

shutil.make_archive("t5-summarization-model", "zip", "./t5-summarization-model")
files.download("t5-summarization-model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>